In [ ]:
import os
import sys

import pandas as pd

#import the same config and metrics the solver uses, so neither the data
#nor the evaluation semantics can drift from the model
sys.path.insert(0, os.path.abspath(".."))
from src.infrastructure.config import DEFAULT_CONFIG as config
from src.core.metrics import compute_metrics, summary_lines

In [ ]:
print(f"{config.num_employees} employees, {config.num_days} days, "
      f"{len(config.weekend_indices)} weekend days")
print("shifts:", config.shifts, "| free:", config.free_shifts)
print("composites:", config.composite_shifts)

In [ ]:
df = pd.read_csv(os.path.join("..", "output", "schedule.csv"))

#column 0 is the employee id, so day d lives in column d + 1
schedule = {
    int(row.iloc[0]): list(row.iloc[1:])
    for _, row in df.iterrows()
}
df.shape

In [ ]:
#the same evaluation the solver logs after each run
metrics = compute_metrics(schedule, config)
for line in summary_lines(metrics, config):
    print(line)

In [ ]:
#weekend days worked per employee; Komp counts as free, like R
worked = pd.Series(metrics.weekend_days_worked)
print(f"fair share: {config.fair_weekend_days}, cap: {config.max_weekend_days_worked}")
worked.value_counts().sort_index()

In [ ]:
#shift slots per employee, weighted: a 1N3N day fills two slots
fair_min, fair_max = config.fair_shifts
slots = pd.Series(metrics.shifts_worked)
print(f"fair band: {fair_min} to {fair_max}, "
      f"outside: {len(metrics.employees_outside_fair_band)} employees")
slots.value_counts().sort_index()

In [ ]:
#raw view of the weekend columns for eyeballing
df.iloc[:, [i + 1 for i in config.weekend_indices]]